## content

消息的content 可以理解为数据内容，它是弱类型的，支持字符串和列表（列表元素通常为字典）

如果需要发送的不只是文本，如多模态内容，则需要content的字典内容遵循模型供应商的API规范，以字典列表形式。openai: gpt-4.1 为例。

参考官方文档：
https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create

In [ ]:
### OpenAI模型使用content字典发送多模态内容-成功这是OpenAI推荐的方式
import base64
import mimetypes
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os
load_dotenv(override=True)
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)


def encode_image(img_path):
    """将本地图片转换成 OpenAI-compatible image_url 可用的 Data URI。"""
    # Data URI 前缀里的 MIME 类型必须和真实图片格式一致。
    # 例如 test_image.png 必须写成 data:image/png，不能伪装成 data:image/jpeg。
    mime_type, _ = mimetypes.guess_type(img_path)
    if mime_type is None or not mime_type.startswith("image/"):
        raise ValueError(f"无法识别图片类型: {img_path}")

    with open(img_path, "rb") as img_file:
        encoded = base64.b64encode(img_file.read()).decode("utf-8")

    return f"data:{mime_type};base64,{encoded}"


# 图像路径
img_path = "test_image.png"
# 获取图像 Data URI 字符串，例如 data:image/png;base64,...
base64_image = encode_image(img_path)
response = model.invoke(
    [
        HumanMessage(
            content=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image_url',
                    # OpenAI-compatible 的 image_url 字段要求是 {"url": ...} 结构。
                    "image_url": {"url": base64_image},
                },
            ]
        )
    ]
)
print(response.content)

图里是一丛正在盛开的白色小雏菊/菊花，花心是黄色的，周围是绿色叶子。画面里还能看到一只小蜜蜂正在花上采蜜。


In [ ]:
### Claude模型使用content字典发送多模态内容-失败Claude有其自己的方式（但实际上成功了，可能新版本做了Block转化）

import base64
import mimetypes
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os
load_dotenv(override=True)
# 这里通过 OpenRouter 调 Claude，所以 provider 仍然选择 openai。
# 原因是 OpenRouter 暴露的是 OpenAI-compatible 接口，不是 Anthropic 原生接口。
model = init_chat_model(
    model="anthropic/claude-haiku-4.5",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)


def encode_image(img_path):
    """将本地图片转换成 OpenAI-compatible image_url 可用的 Data URI。"""
    # Data URI 前缀里的 MIME 类型必须和真实图片格式一致。
    # 例如 test_image.png 必须写成 data:image/png，不能伪装成 data:image/jpeg。
    mime_type, _ = mimetypes.guess_type(img_path)
    if mime_type is None or not mime_type.startswith("image/"):
        raise ValueError(f"无法识别图片类型: {img_path}")

    with open(img_path, "rb") as img_file:
        encoded = base64.b64encode(img_file.read()).decode("utf-8")

    return f"data:{mime_type};base64,{encoded}"


# 图像路径
img_path = "test_image.png"
# 获取图像 Data URI 字符串，例如 data:image/png;base64,...
base64_image = encode_image(img_path)
response = model.invoke(
    [
       HumanMessage(
            content=[
                {"type": "text", "text": "这张图里有什么？"},
                {
                    "type": "image_url",
                    # 虽然底层会转给 Anthropic，但这里仍然经过 OpenRouter 的 OpenAI-compatible 接口。
                    # 因此先按 OpenAI 的 image_url 格式传入，再由供应商适配到 Claude。
                    "image_url": {"url": base64_image},
                },
            ]
        )
    ]
)
print(response.content)

这张图里有：

1. **菊花/雏菊** - 主要花卉，白色花瓣配黄色花心，花朵饱满绽放

2. **蜜蜂** - 在左下方的花朵上采蜜，可以清楚看到

3. **绿叶** - 深绿色的植物叶片作为背景

4. **花蕾** - 尚未开放的黄色小花蕾散布其中

5. **蜘蛛网或虫子** - 在某些花朵上可见细丝

这是一幅生机勃勃的花园微距摄影，展现了菊花的盛开和传粉者与花朵的互动，色彩鲜艳，构图饱满。


## content_blocks

在 LangChain 1.x 中，`content_blocks` 是消息对象（BaseMessage）的一项重大升级。它的核心目标是提供一种跨模型供应商、标准化的多模态数据结构。

过去，处理图片、音频、甚至是模型生成的“思维链（Reasoning）”内容时，不同供应商（OpenAI, Anthropic, Google 等）的 API 格式各异，导致开发者需要写大量的适配代码。`content_blocks` 的出现终结了这种混乱。

在 LangChain 1.2 版本中，消息对象的 `content` 属性依然存在（为了向前兼容），但新增了 `content_blocks` 属性，可以将 `content` 解析为标准、类型安全的表示。
- 数据结构：它是一个 `list[TypedDict]`。
- 统一格式：每个 block 都有一个 `type` 字段，用于区分内容类型。
- 支持类型：包括 `text`（文本）、`image`（图片）、`audio`（音频）、`video`（视频）、`tool_call`（工具调用）以及 `reasoning`（推理/思维链）。

支持的字段类型详见 https://docs.langchain.com/oss/python/langchain/messages#openai

### 输入格式化
对于复杂的对话（带图片或工具结果），建议使用 `content_blocks` 列表形式构建 `HumanMessage` 或 `AIMessage`。

借助 `content_blocks`，我们可以用一套标准代码，无缝地在不同厂商的模型之间切换。

In [4]:
### 使用content_blocks接受多模态输入
import base64
from langchain.messages import HumanMessage
import os
from langchain.chat_models import init_chat_model


from dotenv import load_dotenv
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串，方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

# 图像路径
img_path = "test_image.png"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)

response = model.invoke(
    [
        # 推荐的统一写法
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/png',
                }
            ]
        )
    ]
)
print(response.content)


图里是一片开得很茂盛的白色小花，花瓣偏白、花心是黄色的，像雏菊或菊花一类的花。  
画面里还有很多绿色叶子，并且能看到一只小蜜蜂停在左下方的一朵花上。


### 输出格式化

在之前的案例中DeepSeek模型使用extra_body参数开启思考模式后，思考内容的输出位于additional_kwargs的reasoning_content字段下。其他思考模型的内容可能位于不同的字段下，使用content_blocks可以格式化这些输出。

In [5]:
### DeepSeek使用content_blocks输出思考内容

from langchain.chat_models import init_chat_model
import os 
from dotenv import load_dotenv
load_dotenv(override=True)

model = init_chat_model(model="deepseek-v4-flash")
response = model.invoke("你好，一句话介绍你自己")
response.content_blocks


[{'type': 'reasoning',
  'reasoning': '好的，用户让我用一句话介绍自己。这是一个非常简单直接的要求。我需要用最精炼的语言概括自己的核心功能和特点。我的核心身份是AI助手，主要能力是提供信息、解答问题、帮助创作和进行对话。关键是要突出这些点，同时保持句子流畅自然。想到了可以说“我是DeepSeek，一个由深度求索公司创造的AI助手，致力于为你提供信息、解答问题和协助创作。”这样既说明了身份来源，也明确了服务宗旨。'},
 {'type': 'text', 'text': '你好，我是DeepSeek，由深度求索公司创造的AI助手，乐于为你解答问题、提供信息和协助创作！'}]

### 流式输出格式化

`invoke()` 会等模型完整生成后返回一个 `AIMessage`；`stream()` 会在生成过程中不断返回 `AIMessageChunk`。

每个 chunk 也可以读取 `content_blocks`，所以流式输出时同样可以用统一的标准字段区分 `reasoning`、`text` 等内容。

如果后续还需要完整消息，可以把所有 chunk 累加起来，最后再读取 `full.content_blocks`。


In [6]:
### DeepSeek使用stream + content_blocks流式输出思考内容

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model(model="deepseek-v4-flash")

full = None
printed_reasoning_title = False
printed_answer_title = False

for chunk in model.stream("你好，一句话介绍你自己"):
    # stream() 每次返回一个 AIMessageChunk，代表当前刚生成出来的一小段内容。
    # 把 chunk 累加起来，可以在流式结束后得到完整消息。
    full = chunk if full is None else full + chunk

    # chunk.content_blocks 会把不同供应商的输出统一成标准块。
    # 思考模型可能流出 reasoning 块；普通回答文本通常是 text 块。
    for block in chunk.content_blocks:
        if block["type"] == "reasoning" and block.get("reasoning"):
            if not printed_reasoning_title:
                print("[reasoning]", end=" ", flush=True)
                printed_reasoning_title = True
            print(block["reasoning"], end="", flush=True)

        elif block["type"] == "text" and block.get("text"):
            if not printed_answer_title:
                print("\n\n[answer]", end=" ", flush=True)
                printed_answer_title = True
            print(block["text"], end="", flush=True)

print("\n\n完整消息的 content_blocks：")
full.content_blocks


[reasoning] 好的，用户让我用一句话介绍自己。这是一个非常简单的请求，需要简洁明了地概括我的核心身份和功能。我是DeepSeek，由深度求索公司开发的AI助手，主要特点是免费、支持联网和长上下文。可以用“你好，我是DeepSeek，一个免费、支持联网和超长上下文的AI助手，随时为你提供帮助”来回应，这样既点明了身份，也突出了关键优势。

[answer] 你好，我是DeepSeek，一个免费、支持联网和超长上下文的AI助手，随时为你提供帮助！

完整消息的 content_blocks：


[{'type': 'reasoning',
  'reasoning': '好的，用户让我用一句话介绍自己。这是一个非常简单的请求，需要简洁明了地概括我的核心身份和功能。我是DeepSeek，由深度求索公司开发的AI助手，主要特点是免费、支持联网和长上下文。可以用“你好，我是DeepSeek，一个免费、支持联网和超长上下文的AI助手，随时为你提供帮助”来回应，这样既点明了身份，也突出了关键优势。'},
 {'type': 'text', 'text': '你好，我是DeepSeek，一个免费、支持联网和超长上下文的AI助手，随时为你提供帮助！'}]